In [ ]:
from collections import defaultdict
from glob import glob
from pathlib import Path
import os
import warnings
warnings.simplefilter("ignore", UserWarning)

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors
import numpy as np
from shapely.ops import transform

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"
targets_path = "power/targets.geoparquet"  # within results_dir
country_polygons_path = "input/admin-boundaries/admin-level-0.geoparquet"  # within results_dir

# Output paths
plot_dir = Path("../plots")

# Mapping from region display name to region bounding box
regions = {
    "Americas": [-113, -3, -57, 53],
    "Indian Ocean": [36, -29, 99, 34],
    "West Pacific": [100, -53, 155, 57],
}
# Mapping from open-gira STORM_SET to display name (future STORM will be aggregated)
atmospheres_to_scenario = {
    "STORM_constant": "STORM 2020",
    "STORM_CNRM-CM6-1-HR": "STORM 2050 RCP8.5",
    "STORM_CMCC-CM2-VHR4": "STORM 2050 RCP8.5",
    "STORM_EC-Earth3P-HR": "STORM 2050 RCP8.5",
    "STORM_HadGEM3-GC31-HM": "STORM 2050 RCP8.5",
    "IRIS_PRESENT": "IRIS 2020",
    "IRIS_SSP5-2050": "IRIS 2050 SSP5-8.5"
}
# Scenarios to plot
scenarios = list(sorted(set(atmospheres_to_scenario.values())))

# Region aspect ratios must be appropriate for plot layout, i.e. square for a square plot
# Absolute sizes of bbox do not matter
print("Region,\t\txspan, yspan, ratio")
for region, bbox in regions.items():
    xmin, ymin, xmax, ymax = bbox
    xspan = xmax - xmin
    yspan = ymax - ymin
    print(f"{region},\t{xspan:.2f}, {yspan:.2f}, {xspan/yspan:.2f}")

In [ ]:
# Estimate population connected to grid in each country
targets = gpd.read_parquet(os.path.join(results_dir, targets_path), columns=["population", "iso_a3", "geometry"])
targets["geometry"] = targets.geometry.representative_point()

# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)
# Global default threshold, ms-1
default_threshold = 30.0

atmosphere_to_disruption = []
for atmosphere in atmospheres_to_scenario.keys():
    path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/EAPA_admin-level-2-0.gpq")
    df = gpd.read_parquet(path)
    
    # Set countries with no calibrated value to the global default
    df["pop_at_risk"] = df[f"{default_threshold:.1f}"]
    
    valid_countries = thresholds.index.unique()
    df = df[df.ISO_A3.isin(valid_countries)]
    df = df.merge(
        thresholds[['threshold_ms-1']],
        left_on='ISO_A3',
        right_index=True,
        how='left'
    )
    
    df['pop_at_risk'] = df.apply(
        lambda row: row[str(row['threshold_ms-1'])] if pd.notna(row['threshold_ms-1']) else np.nan,
        axis=1
    )
        
    # Drop the now redundant master set of thresholds
    cols_to_keep = ['GID_0', 'GID_1', 'GID_2', 'ISO_A3', 'NAME_0', 'NAME_1', 'NAME_2', 'geometry', 'pop_at_risk']
    
    # Ideally, I wouldn't fillna here, but if I remove this (and handle the zero division error later)
    # the results are not what I'd expect... so I'm leaving it and not investigating any further for now
    df["pop_at_risk"] = df["pop_at_risk"].fillna(0)
    
    df = df.loc[:, cols_to_keep].reset_index(drop=True)
    
    df = df.sort_values("pop_at_risk", ascending=False)
    
    df["atmosphere"] = atmosphere
    df["GID"] = df.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
    atmosphere_to_disruption.append(df)
    
# all_atmospheres = pd.concat(atmosphere_to_disruption)
# all_atmospheres["scenario"] = all_atmospheres.atmosphere.map(atmospheres_to_scenario)

# storm_future_mask = all_atmospheres.scenario == "STORM 2050 RCP8.5"
# storm_future_pop_at_risk = all_atmospheres.loc[storm_future_mask, ["GID", "pop_at_risk"]].groupby("GID").mean()
# storm_future_mean = all_atmospheres[storm_future_mask].drop_duplicates("GID") \
#     .drop(columns=["pop_at_risk", "atmosphere"]).set_index("GID").join(storm_future_pop_at_risk)
# concat = pd.concat([all_atmospheres[~storm_future_mask], storm_future_mean])
# concat["GID"] = concat.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
# df = concat.sort_values(["GID", "scenario"]).set_index("GID") 

# # Duplicate regions, each with different target
# region_to_target = df.sjoin(targets, how="left")
# # Group by region, sum target populations to find connected population per region
# population = region_to_target[["population"]].groupby(region_to_target.index).sum()
# df = df.join(population)

# # Calculate a measure normalised by the regional (connected) population
# df["pop_at_risk_pc"] = df.pop_at_risk / df.population

# # We can't log plot a zero value, so nudge 0's into positive
# nudge = 1E-9
# df.loc[df.pop_at_risk_pc == 0, "pop_at_risk_pc"] = nudge
# df.loc[df.pop_at_risk == 0, "pop_at_risk"] = nudge

# # Determine whether network representations pass muster
# node_paths = glob(os.path.join(results_dir, "power/by_country/*/network/nodes.geoparquet"))
# iso_a3_grid_state = {}
# for path in node_paths:
#     iso_a3 = path.split("/")[-3]
#     print(iso_a3, end="\r")
    
#     try:
#         nodes = pd.read_parquet(path, columns=["asset_type", "power_mw"])
#     except ValueError:
#         # empty node file
#         iso_a3_grid_state[iso_a3] = False
#         continue

#     source_nodes = nodes[nodes.asset_type == "source"]
#     target_nodes = nodes[nodes.asset_type == "target"]
#     operating = ((source_nodes.power_mw.sum() > 0) and (target_nodes.power_mw.sum() < 0))
#     iso_a3_grid_state[iso_a3] = operating
    
# grids = pd.Series(index=iso_a3_grid_state.keys(), data=iso_a3_grid_state.values(), name="operating")
# bad_grids = grids[~grids].index

# # Set results for bad grids to NaN
# df.loc[df.ISO_A3.isin(bad_grids), ["pop_at_risk", "pop_at_risk_pc"]] = np.nan

# # If any countries are missing from our dataset, add them so we can explicitly mark as missing
# countries = gpd.read_parquet(os.path.join(results_dir, country_polygons_path))
# countries.geometry = countries.geometry.boundary
# missing_countries = list(set(countries.GID_0) - set(df.ISO_A3))
# # The missing countries will have NaN valued pop_at_risk and pop_at_risk_pc 
# df = pd.concat([df, countries.set_index("GID_0").loc[missing_countries]])

# df = df.set_index([df.index, df.scenario])

In [ ]:
def shift_lon(geom):
    def shift(x, y, z=None):
        x = np.where(x < 0, x + 360, x)
        return x, y
    return transform(shift, geom)

df["geometry"] = df["geometry"].apply(shift_lon)
df = df[~df.atmosphere.isna()]

In [ ]:
fig = plt.figure(figsize=(10, 5))
cmap = "Blues"
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
#ax.set_extent([0, 360, -90, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.9")
ax.add_feature(cfeature.OCEAN, facecolor="0.97")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
norm = matplotlib.colors.LogNorm(
    vmin=1E2,
    vmax=df.pop_at_risk.quantile(0.95),
)
df.plot(
    "pop_at_risk",
    transform=ccrs.PlateCarree(),
    ax=ax,
    cmap=cmap,
    norm=norm,
    legend=False,
)
cax = fig.add_axes([1.05, 0.2, 0.03, 0.6])
sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Population at risk')